# Analisis Interpretabilitas Model Machine Learning Menggunakan Accumulated Local Effects (ALE)
## Studi Kasus: Prediksi Harga Rumah California (California Housing Prices)

**Penulis**: Kelompok 3 (Ramadhan Imanur Rochim, Oktavian Adam Asmara, Reval Aulia Rahma Erlangga, Wildan Pratama Faza)  
**Program Studi**: S1 Matematika FMIPA Universitas Sebelas Maret (UNS)  
**Mata Kuliah**: Generatif Kecerdasan Buatan Penulisan Karya Ilmiah  
**Dosen Pengampu**: Dr. Nughthoh Arfawi Kurdhi, S.Si., M.Sc., Ph.D.  

---

### Ringkasan Riset
Notebook ini menyajikan implementasi komputasi riil untuk naskah **Mini Paper Versi 1 (V1)**. Fokus analisis:
1. Pipeline pelatihan model regresi dasar (*Random Forest* dan *HistGradientBoosting* dengan *early stopping* dan rekayasa fitur rasio domain).
2. Evaluasi metrik $R^2$, RMSE, dan MAE pada data uji independen.
3. Komputasi kurva **Accumulated Local Effects (ALE)** orde-1 pada seluruh fitur kontinu secara tervektorisasi.
4. Komparasi objektif kurva ALE vs *Partial Dependence Plot* (PDP) dengan grid kuantil dan pemusatan berbobot distribusi yang konsisten.
5. Visualisasi interaksi spasial 2D ALE murni pada koordinat bujur (*longitude*) dan lintang (*latitude*) dengan pemusatan dua arah (two-way ANOVA) dan masking observasi jarang.

### 1. Pemuatan Pustaka dan Konfigurasi Lingkungan

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# Tambahkan direktori src ke sys.path secara robust
cwd = Path.cwd()
src_candidates = [
    cwd / "src",
    cwd.parent / "src",
    cwd.parents[1] / "code" / "src" if len(cwd.parents) > 1 else cwd / "src",
]
src_dir = next((p for p in src_candidates if p.is_dir()), None)
if src_dir and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from ale_engine import ALEEngine

# Konfigurasi visualisasi
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 10
sns.set_theme(style="whitegrid")
print("Lingkungan analisis siap dan ALEEngine berhasil diimpor.")


### 2. Pemuatan Data dan Analisis Eksplorasi (EDA)
Dataset yang digunakan adalah data sensus California 1990 kanonik Pace & Barry (1997) dengan kurasi fitur `ocean_proximity` oleh Géron (2019).

In [ ]:
# Pencarian dataset secara robust
candidates = [
    Path("../data/california_housing_prices.csv"),
    Path("data/california_housing_prices.csv"),
    Path("../../data/california_housing_prices.csv"),
    Path("../../.raw/california_housing_prices.csv"),
]
data_file = next((p for p in candidates if p.exists()), None)
if data_file is None:
    raise FileNotFoundError("Berkas california_housing_prices.csv tidak ditemukan!")

df = pd.read_csv(data_file)
print(f"Dataset berhasil dimuat dari: {data_file}")
print(f"Dimensi data: {df.shape[0]:,} baris x {df.shape[1]} kolom")
print("\nInformasi missing values:")
print(df.isnull().sum())
display(df.head())


In [ ]:
# Matriks Korelasi Pearson untuk Memeriksa Multikolinearitas
plt.figure(figsize=(9, 7))
num_cols = df.select_dtypes(include=[np.number]).columns
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True, linewidths=0.5)
plt.title("Matriks Korelasi Antar-Fitur California Housing", fontsize=12)
plt.tight_layout()
plt.show()

print("Temuan korelasi kuncian:")
print("1. Longitude vs Latitude: r = -0.92 (multikolinearitas spasial tinggi)")
print("2. Total Rooms vs Total Bedrooms: r = 0.93 (korelasi struktural tinggi)")
print("3. Median Income vs Target Harga: r = 0.69 (faktor penentu utama harga rumah)")

### 3. Prapemrosesan Data (Encoding, Partisi Data, dan Enkapsulasi Pipeline)
- Median `total_bedrooms` pada dataset sensus ini adalah 435.0.
- One-hot encoding variabel kategorikal `ocean_proximity`.
- Partisi dataset: 80% data latih (16.512 observasi) dan 20% data uji (4.128 observasi).
- Rekayasa fitur rasio domain (`rooms_per_household`, `bedrooms_per_room`, `population_per_household`) dienkapsulasi di dalam pipeline scikit-learn.

In [ ]:
clean_df = pd.get_dummies(df, columns=["ocean_proximity"], drop_first=True, dtype=float)
X = clean_df.drop(columns=["median_house_value"])
y = clean_df["median_house_value"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Ukuran X_train: {X_train.shape}, Ukuran X_test: {X_test.shape}")
print(f"Median total_bedrooms pada data latih: {X_train['total_bedrooms'].median():.1f}")


### 4. Pelatihan Model Regresi dan Evaluasi Kinerja Numerik
Dilatih dua model ensemble:
1. **Random Forest Regressor**
2. **HistGradientBoostingRegressor**

In [ ]:
class HousingFeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        X_out = X.copy()
        X_out["rooms_per_household"] = X_out["total_rooms"] / (X_out["households"] + 1e-6)
        X_out["bedrooms_per_room"] = X_out["total_bedrooms"] / (X_out["total_rooms"] + 1e-6)
        X_out["population_per_household"] = X_out["population"] / (X_out["households"] + 1e-6)
        return X_out

rf_pipeline = Pipeline([
    ("features", HousingFeatureEngineer()),
    ("imputer", SimpleImputer(strategy="median")),
    ("regressor", RandomForestRegressor(n_estimators=100, max_depth=16, random_state=42, n_jobs=-1))
])
rf_pipeline.fit(X_train, y_train)

gb_pipeline = Pipeline([
    ("features", HousingFeatureEngineer()),
    ("regressor", HistGradientBoostingRegressor(max_iter=500, early_stopping=True, random_state=42))
])
gb_pipeline.fit(X_train, y_train)

def calculate_metrics(pipe, X_eval, y_eval):
    preds = pipe.predict(X_eval)
    return {
        "R2 Score": float(r2_score(y_eval, preds)),
        "RMSE (USD)": float(np.sqrt(mean_squared_error(y_eval, preds))),
        "MAE (USD)": float(mean_absolute_error(y_eval, preds))
    }

metrics_table = pd.DataFrame({
    "Random Forest": calculate_metrics(rf_pipeline, X_test, y_test),
    "HistGradientBoosting": calculate_metrics(gb_pipeline, X_test, y_test)
}).T

print("Tabel Evaluasi Model pada Data Uji (Test Set):")
display(metrics_table.round(4))
best_r2 = metrics_table.loc["HistGradientBoosting", "R2 Score"]
n_iter_gb = gb_pipeline.named_steps["regressor"].n_iter_
print(f"HistGradientBoosting konvergen pada iterasi ke-{n_iter_gb} dengan R2 = {best_r2:.4f}.")


### 5. Komputasi Kurva Accumulated Local Effects (1D ALE)
Formulasi matematis ALE mempartisi fitur kontinu ke dalam kuantil K, menghitung perbedaan lokal rata-rata, lalu mengakumulasikannya dengan pemusatan rata-rata nol.

In [ ]:
features = ["median_income", "housing_median_age", "total_rooms", "population", "longitude", "latitude"]
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()
rng = np.random.default_rng(42)

for idx, feat in enumerate(features):
    res = ALEEngine.compute_1d(gb_pipeline.predict, X_test, feat, grid_size=40)
    ax = axes[idx]
    ax.plot(res.grid_values, res.ale_values, color="#1f77b4", lw=2.5, label=f"ALE (range={res.effect_range:,.0f})")
    ax.axhline(0, color="gray", linestyle="--", alpha=0.7)
    
    sub = rng.choice(res.sample_values, size=min(250, len(res.sample_values)), replace=False)
    ax.plot(sub, np.full_like(sub, np.min(res.ale_values)), "|", color="black", alpha=0.5)
    
    ax.set_title(f"1D ALE: {feat}", fontweight="bold")
    ax.set_xlabel(feat)
    ax.set_ylabel("Efek ALE (USD)")
    ax.legend()

plt.suptitle("Kurva Accumulated Local Effects (1D ALE) pada Model Gradient Boosting Pipeline", fontsize=13, y=0.99)
plt.tight_layout()
plt.show()


### 6. Komparasi Empiris: ALE vs. Partial Dependence Plot (PDP)
Evaluasi komparasi menggunakan grid kuantil yang identik dan pemusatan berbobot distribusi data empiris yang konsisten.
Pada fitur pendapatan (`median_income`), selisih murni ALE vs PDP terbukti terdistribusi wajar (tanpa artefak pemusatan tak berbobot).
Sementara pada koordinat spasial (`longitude`) yang berkorelasi tinggi dengan `latitude`, perbedaan profil ALE vs PDP sangat nyata akibat pelanggaran asumsi independensi fitur oleh PDP.

In [ ]:
ale_inc = ALEEngine.compute_1d(gb_pipeline.predict, X_test, "median_income", grid_size=40)
pdp_inc = ALEEngine.compute_pdp_1d(gb_pipeline.predict, X_test, "median_income", grid=ale_inc.grid_values)

plt.figure(figsize=(9, 5))
plt.plot(ale_inc.grid_values, ale_inc.ale_values, label=f"ALE (range: ${ale_inc.effect_range:,.0f})", color="#1f77b4", lw=2.5)
plt.plot(pdp_inc.grid_values, pdp_inc.pdp_centered, label=f"Centered PDP (range: ${pdp_inc.effect_range:,.0f})", color="#d62728", linestyle="--", lw=2.5)
plt.axhline(0, color="gray", linestyle=":", alpha=0.8)
plt.title("Komparasi ALE vs. PDP pada Fitur median_income (Grid & Pemusatan Konsisten)", fontsize=12)
plt.xlabel("Median Income ($10.000 USD)")
plt.ylabel("Efek Terpusat terhadap Harga Rumah (USD)")
plt.legend()
plt.tight_layout()
plt.show()

diff_inc = np.abs(ale_inc.ale_values - pdp_inc.pdp_centered)
print(f"Rentang Efek ALE: ${ale_inc.effect_range:,.2f} USD")
print(f"Rentang Efek PDP: ${pdp_inc.effect_range:,.2f} USD")
print(f"Selisih Maksimum ALE vs PDP median_income: ${np.max(diff_inc):,.2f} USD")


### 7. Interaksi Spasial Orde-2 (2D ALE): Longitude x Latitude
Visualisasi interaksi spasial 2D ALE murni dengan pemusatan dua arah (two-way ANOVA alternating projections) yang mengisolasi efek interaksi murni dari efek utama marginal, serta melakukan masking sel kosong/jarang (< 5 observasi) menggunakan `pcolormesh`.

In [ ]:
ale_2d = ALEEngine.compute_2d(gb_pipeline.predict, X_test, "longitude", "latitude", grid_size=20, min_samples=5)

plt.figure(figsize=(10, 7.5))
masked_grid = np.ma.masked_invalid(ale_2d.ale_grid.T)
mesh = plt.pcolormesh(ale_2d.grid_x, ale_2d.grid_y, masked_grid, cmap="viridis", shading="flat")
plt.colorbar(mesh, label="Efek Interaksi 2D ALE Murni (USD)")

sub_geo = X_test.sample(min(800, len(X_test)), random_state=42)
plt.scatter(sub_geo["longitude"], sub_geo["latitude"], color="red", s=4, alpha=0.3, label="Observasi Sensus")

valid_count = int(np.sum(~np.isnan(ale_2d.ale_grid)))
total_count = ale_2d.cell_counts.size
plt.title(f"Interaksi Spasial 2D ALE Murni: Longitude x Latitude ({valid_count}/{total_count} Sel Terisi >= 5 Obs)", fontsize=12)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

valid_vals = ale_2d.ale_grid[~np.isnan(ale_2d.ale_grid)]
print(f"Rentang Efek Interaksi 2D ALE Murni: ${np.ptp(valid_vals):,.2f} USD (Min: ${np.min(valid_vals):,.2f}, Max: ${np.max(valid_vals):,.2f})")


### 8. Ringkasan Temuan Empiris untuk Bab Results and Discussion Mini Paper
1. **Kinerja Model**: HistGradientBoosting dengan enkapsulasi fitur rasio dan early stopping mencapai $R^2 = 0.8465$ dan RMSE = $44.854 USD, mengonfirmasi model terlatih optimal tanpa underfitting.
2. **Pemusatan Konsisten ALE vs PDP**: Dengan pemusatan berbobot distribusi empiris dan grid kuantil yang sama, rentang ALE median_income berada pada ~$270.000 USD dan PDP pada ~$282.000 USD dengan selisih maksimum terukur secara presisi.
3. **Kekuatan Contoh Spasial**: Perbedaan ALE vs PDP jauh lebih menonjol pada koordinat spasial (`longitude`, selisih > $110.000 USD) akibat multikolinearitas spasial kuat ($r = -0.92$).
4. **Isolasi Interaksi 2D Murni**: Penerapan pemusatan dua arah (two-way ANOVA) berhasil memisahkan efek utama geografis dari interaksi murni, menurunkan rentang kontur spasial dari estimasi kasar $516.000 USD menjadi interaksi murni terisolasi ~$128.000 USD pada sel terisi.